# FunctionGemma TPU Fine-Tuning with TorchTPU & LiteRT-LM Deployment

<table class="tfo-notebook-buttons">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

[LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) is Google’s high-performance edge inference engine that runs LLMs across mobile, web, and desktop with full CPU/GPU/NPU acceleration and production-ready SDKs.

This notebook fine-tunes [FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it) on a single Colab TPU using [TorchTPU + TorchTitan](https://github.com/google-pytorch/torchtitan/tree/main/torchtitan/experiments/tpu), then converts the model to `.litertlm` (`llm_model_type: function_gemma`) for on-device tool-calling inference.

- **Part A — TPU Fine-Tuning**: Teaches FunctionGemma 270M a custom smart-home tool set (canonical IDs such as `light.living_room` and `climate.bedroom`, plus numeric brightness/temperature arguments) using prompts rendered exactly as LiteRT-LM renders them on device.
- **Part B — LiteRT-LM Export & Evaluation**: Exports the Base and Fine-Tuned models to `.litertlm` with `litert_lm_model_type_override="function_gemma"` (`FunctionGemmaDataProcessor`) and measures tool-call accuracy on held-out requests (unseen phrasings and devices) in `litert_lm.Engine`.

**Setup**:
- Connect to either a **`v5e-1`** or **`v6e-1`** TPU runtime (**Runtime $\rightarrow$ Change runtime type $\rightarrow$ TPU**).
- **Estimated Runtime**: Expect **~10+ minutes** end-to-end when using a `v5e-1` TPU.
- Add `HF_TOKEN` in Colab **Secrets (🔑)** if downloading the gated `google/functiongemma-270m-it` repository or uploading checkpoints to Hugging Face Hub.

## Part A — Fine-Tuning on TPU (`TorchTPU` + `TorchTitan`)

### Step 1: Install `torch_tpu` + `torchtitan` & Verify TPU

In [ ]:
# @title { display-mode: "form" }
import glob
import importlib.metadata as md
import os
import subprocess
import sys

TORCHTITAN_DIR = "/content/torchtitan"

if not os.path.exists(TORCHTITAN_DIR):
    print("Cloning https://github.com/google-pytorch/torchtitan.git (main) ...")
    subprocess.run(
        ["git", "clone", "https://github.com/google-pytorch/torchtitan.git", TORCHTITAN_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", TORCHTITAN_DIR, "pull", "--ff-only"], check=False)

abi_pins = []
for pkg in ("numpy", "pyarrow", "pillow"):
    try:
        abi_pins.append(f"{pkg}=={md.version(pkg)}")
    except Exception:
        pass

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "keyrings.google-artifactregistry-auth"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.15.0", "torch_tpu==0.2.0",
        "-r", f"{TORCHTITAN_DIR}/torchtitan/experiments/tpu/requirements.txt",
        "--extra-index-url", "https://download.pytorch.org/whl/test/cpu",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{TORCHTITAN_DIR}/requirements.txt"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"],
    check=False,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "--upgrade-strategy", "only-if-needed",
        "transformers>=5.17.0", "accelerate>=1.2.0",
        "huggingface_hub>=0.28.0", "safetensors>=0.4.5",
        *abi_pins,
    ],
    check=True,
)

vfio_devs = [p for p in glob.glob("/dev/vfio/[0-9]*") if os.path.basename(p).isdigit()]
accel_devs = glob.glob("/dev/accel[0-9]*")
detected_chips = len(vfio_devs) or len(accel_devs) or 1

# Verify TPU in a subprocess so the notebook kernel does not lock /dev/vfio/0 before torchrun
verify_cmd = (
    "import importlib.metadata as md, torch, torch_tpu; "
    f"cnt = torch.accelerator.device_count() if hasattr(torch, 'accelerator') else {detected_chips}; "
    "print('[OK] PyTorch=' + str(torch.__version__) + ' | torch_tpu=' + str(md.version('torch_tpu')) + f' | Visible TPU chips: {cnt}')"
)
res = subprocess.run([sys.executable, "-c", verify_cmd], capture_output=True, text=True)
print(res.stdout if res.returncode == 0 else f"[INFO] Detected TPU device nodes: {detected_chips}")

### Step 2: Stage Base Model & Build the Smart-Home Tool-Calling Dataset

Downloads `google/functiongemma-270m-it` and generates synthetic smart-home requests for three tools (`set_device_state`, `set_brightness`, `set_temperature`). The targets use canonical IDs (`light.living_room`, `climate.bedroom`) and numeric arguments that the base model has not been trained on.

- Prompts are rendered exactly as LiteRT-LM's `FunctionGemmaDataProcessor` renders them at inference (tool declarations in the `developer` turn), so training matches on-device inference.
- Some devices/rooms (e.g. the living-room lights) and phrasings appear **only** in the held-out evaluation set used in Step 7.

In [ ]:
# @title { display-mode: "form" }
import json
import os
import random
from huggingface_hub import login, snapshot_download

MODEL_ID = "google/functiongemma-270m-it"
FALLBACK_MODEL_ID = "unsloth/functiongemma-270m-it"
BASE_MODEL_DIR = "/content/base_functiongemma_hf"
TRAIN_ASSETS_DIR = "/content/functiongemma_train_assets"
SFT_DATA_DIR = "/content/sft_data"
NUM_TRAIN = 1500
NUM_EVAL = 300

os.makedirs(BASE_MODEL_DIR, exist_ok=True)
os.makedirs(SFT_DATA_DIR, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

if not os.path.exists(os.path.join(BASE_MODEL_DIR, "config.json")):
    try:
        print(f"Downloading {MODEL_ID} to {BASE_MODEL_DIR} ...")
        snapshot_download(repo_id=MODEL_ID, local_dir=BASE_MODEL_DIR, token=hf_token)
    except Exception as e:
        print(f"[INFO] Could not download gated {MODEL_ID} ({e}); falling back to {FALLBACK_MODEL_ID} ...")
        snapshot_download(repo_id=FALLBACK_MODEL_ID, local_dir=BASE_MODEL_DIR)
print(f"[OK] Base FunctionGemma 270M checkpoint staged at: {BASE_MODEL_DIR}")

# ---------------------------------------------------------------------------
# 1. Smart-home tools. The model must learn the canonical ID conventions
#    (`light.living_room`, `climate.bedroom`) and numeric arguments.
# ---------------------------------------------------------------------------
TOOLS = [
    {
        "name": "set_device_state",
        "description": "Turn a smart-home device on or off.",
        "parameters": {
            "type": "object",
            "properties": {
                "device_id": {"type": "string", "description": "Device ID formatted as <device_type>.<room>"},
                "state": {"type": "string", "description": "on or off"},
            },
            "required": ["device_id", "state"],
        },
    },
    {
        "name": "set_brightness",
        "description": "Set the brightness of a light.",
        "parameters": {
            "type": "object",
            "properties": {
                "device_id": {"type": "string", "description": "Light ID formatted as light.<room>"},
                "level": {"type": "integer", "description": "Brightness percentage from 0 to 100"},
            },
            "required": ["device_id", "level"],
        },
    },
    {
        "name": "set_temperature",
        "description": "Set the target temperature of a room.",
        "parameters": {
            "type": "object",
            "properties": {
                "zone_id": {"type": "string", "description": "Climate zone ID formatted as climate.<room>"},
                "celsius": {"type": "integer", "description": "Target temperature in degrees Celsius"},
            },
            "required": ["zone_id", "celsius"],
        },
    },
]
with open(os.path.join(SFT_DATA_DIR, "tools.json"), "w", encoding="utf-8") as f:
    json.dump(TOOLS, f, indent=2)


# ---------------------------------------------------------------------------
# 2. Render prompts exactly as LiteRT-LM does at inference time
#    (FunctionGemmaDataProcessor + FunctionGemma chat template), so training
#    and on-device inference see identical text.
# ---------------------------------------------------------------------------
def fc_value(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return f"<escape>{value}<escape>"
    if isinstance(value, (int, float)):
        return json.dumps(value)
    if isinstance(value, list):
        return "[" + ",".join(fc_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{fc_value(v)}" for k, v in value.items()) + "}"
    raise TypeError(f"Unsupported value: {value!r}")


def uppercase_types(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if k == "type" and v in ("string", "number", "integer", "object", "array", "boolean", "null"):
                out[k] = v.upper()
            else:
                out[k] = uppercase_types(v)
        return out
    if isinstance(obj, list):
        return [uppercase_types(v) for v in obj]
    return obj


def fc_declaration(tool: dict) -> str:
    fields = {k: v for k, v in uppercase_types(tool).items() if k != "name"}
    return f"declaration:{tool['name']}{fc_value(fields)}"


DEVELOPER_TURN = (
    "<start_of_turn>developer\n"
    + "".join(f"<start_function_declaration>{fc_declaration(t)}<end_function_declaration>" for t in TOOLS)
    + "<end_of_turn>\n"
)


def render_prompt(query: str) -> str:
    return DEVELOPER_TURN + f"<start_of_turn>user\n{query.strip()}<end_of_turn>\n<start_of_turn>model\n"


def render_call(name: str, args: dict) -> str:
    arg_str = ",".join(f"{k}:{fc_value(v)}" for k, v in args.items())
    return f"<start_function_call>call:{name}{{{arg_str}}}<end_function_call><start_function_response>"


# TorchTitan wraps (question, answer) with the tokenizer's chat template. Use a pass-through
# template in a training-only assets folder so the rendered text above is used verbatim
# (the exported .litertlm still ships the original FunctionGemma template).
os.makedirs(TRAIN_ASSETS_DIR, exist_ok=True)
for fname in os.listdir(BASE_MODEL_DIR):
    src, dst = os.path.join(BASE_MODEL_DIR, fname), os.path.join(TRAIN_ASSETS_DIR, fname)
    if fname == "chat_template.jinja" or not os.path.isfile(src) or os.path.lexists(dst):
        continue
    os.symlink(src, dst)
with open(os.path.join(TRAIN_ASSETS_DIR, "chat_template.jinja"), "w", encoding="utf-8") as f:
    f.write("{{ bos_token }}{%- for message in messages -%}{{ message['content'] }}{%- endfor -%}")

# ---------------------------------------------------------------------------
# 3. Synthetic smart-home requests with held-out rooms/devices and phrasings.
# ---------------------------------------------------------------------------
ROOMS = {
    "living_room": ["living room", "lounge"], "kitchen": ["kitchen"], "bedroom": ["bedroom"],
    "office": ["office", "study"], "garage": ["garage"], "bathroom": ["bathroom"],
    "dining_room": ["dining room"], "hallway": ["hallway", "hall"], "basement": ["basement"],
}
DEVICES = {"light": ["lights", "light", "lamps"], "fan": ["fan", "ceiling fan"], "tv": ["TV", "television"], "plug": ["smart plug", "outlet"]}
HELD_OUT_PAIRS = {("light", "living_room"), ("fan", "bedroom"), ("tv", "office"), ("light", "garage")}
HELD_OUT_CLIMATE = {"living_room", "office"}

STATE_TEMPLATES = {
    "train": {
        "on": ["Turn on the {room} {dev}", "Switch on the {dev} in the {room}", "Please power up the {room} {dev}", "Can you turn the {room} {dev} on?"],
        "off": ["Turn off the {room} {dev}", "Switch off the {dev} in the {room}", "Shut off the {dev} in the {room}", "Please kill the {room} {dev}"],
    },
    "eval": {
        "on": ["I'd like the {room} {dev} on", "Activate the {dev} in the {room}"],
        "off": ["Could you turn the {room} {dev} off", "Deactivate the {dev} in the {room}"],
    },
}
BRIGHTNESS_TEMPLATES = {
    "train": ["Set the {room} {dev} to {pct} percent", "Dim the {room} {dev} to {pct}%", "Change the {dev} in the {room} to {pct} percent brightness"],
    "eval": ["Put the {dev} in the {room} at {pct}%", "I want the {room} {dev} at {pct} percent"],
}
TEMP_TEMPLATES = {
    "train": ["Set the {room} to {t} degrees", "Make it {t} degrees in the {room}", "Heat the {room} to {t} degrees Celsius"],
    "eval": ["I want the {room} at {t} degrees", "Change the {room} temperature to {t}"],
}

rng = random.Random(0)
train_rows, eval_rows, seen_queries = [], [], set()


def add_example(bucket: list, query: str, name: str, args: dict) -> None:
    if query in seen_queries:
        return
    seen_queries.add(query)
    bucket.append({"query": query, "name": name, "arguments": args})


for dev_type, dev_names in DEVICES.items():
    for room_id, room_names in ROOMS.items():
        held_out = (dev_type, room_id) in HELD_OUT_PAIRS
        # Held-out devices only appear in eval, with both training and eval phrasings.
        plans = [("eval", "train"), ("eval", "eval")] if held_out else [("train", "train"), ("eval", "eval")]
        for target, phrasing in plans:
            bucket = train_rows if target == "train" else eval_rows
            for state, templates in STATE_TEMPLATES[phrasing].items():
                for tmpl in templates:
                    add_example(bucket, tmpl.format(room=rng.choice(room_names), dev=rng.choice(dev_names)),
                                "set_device_state", {"device_id": f"{dev_type}.{room_id}", "state": state})
            if dev_type == "light":
                for tmpl in BRIGHTNESS_TEMPLATES[phrasing]:
                    for pct in rng.sample(range(5, 100, 5), 4):
                        add_example(bucket, tmpl.format(room=rng.choice(room_names), dev=rng.choice(dev_names), pct=pct),
                                    "set_brightness", {"device_id": f"light.{room_id}", "level": pct})

for room_id, room_names in ROOMS.items():
    held_out = room_id in HELD_OUT_CLIMATE
    plans = [("eval", "train"), ("eval", "eval")] if held_out else [("train", "train"), ("eval", "eval")]
    for target, phrasing in plans:
        bucket = train_rows if target == "train" else eval_rows
        for tmpl in TEMP_TEMPLATES[phrasing]:
            for t in rng.sample(range(16, 29), 4):
                add_example(bucket, tmpl.format(room=rng.choice(room_names), t=t), "set_temperature", {"zone_id": f"climate.{room_id}", "celsius": t})

# Showcase prompts used in Step 7 -- held out from training (same prompts as the CLI & Web Colab).
DEMO_EXAMPLES = [
    {"query": "Turn on the living room lights", "name": "set_device_state", "arguments": {"device_id": "light.living_room", "state": "on"}},
    {"query": "I want the office at 22 degrees", "name": "set_temperature", "arguments": {"zone_id": "climate.office", "celsius": 22}},
]
demo_queries = {d["query"] for d in DEMO_EXAMPLES}
train_rows = [r for r in train_rows if r["query"] not in demo_queries]
eval_rows = [r for r in eval_rows if r["query"] not in demo_queries]

rng.shuffle(train_rows)
rng.shuffle(eval_rows)
train_rows = train_rows[:NUM_TRAIN]
eval_rows = DEMO_EXAMPLES + eval_rows[: NUM_EVAL - len(DEMO_EXAMPLES)]
train_queries = {r["query"] for r in train_rows}
assert not any(r["query"] in train_queries for r in eval_rows), "Eval queries must not appear in training."

with open(os.path.join(SFT_DATA_DIR, "train.jsonl"), "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps({"question": render_prompt(r["query"]), "answer": render_call(r["name"], r["arguments"])}) + "\n")
with open(os.path.join(SFT_DATA_DIR, "val.jsonl"), "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps({"question": render_prompt(r["query"]), "answer": render_call(r["name"], r["arguments"])}) + "\n")
with open(os.path.join(SFT_DATA_DIR, "eval_toolcalls.jsonl"), "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps(r) + "\n")

print(f"[OK] Smart-home SFT splits: train={len(train_rows)}, held-out eval={len(eval_rows)} (written to {SFT_DATA_DIR})")
print("Sample training target:", train_rows[0]["query"], "->", render_call(train_rows[0]["name"], train_rows[0]["arguments"]))

### Step 3: Verify `hf_sft_full_max` SFT Config

In [ ]:
# @title { display-mode: "form" }
import os
import subprocess
import sys

TORCHTITAN_DIR = "/content/torchtitan"
tpu_exp_dir = os.path.join(TORCHTITAN_DIR, "torchtitan", "experiments", "tpu")

# TorchTitan moved TPU helpers into `distributed/` and `components/` subpackages; support both layouts.
for fname, old_cond, new_cond in [
    ("distributed_utils.py", "if num_devices <= 1:", "if num_devices < 1:"),
    ("cloud_distributed_utils.py", "if world_size == 1:", "if world_size < 1:"),
]:
    for fpath in (os.path.join(tpu_exp_dir, "distributed", fname), os.path.join(tpu_exp_dir, fname)):
        if os.path.isfile(fpath):
            with open(fpath, "r", encoding="utf-8") as f:
                src = f.read()
            if old_cond in src:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(src.replace(old_cond, new_cond))
                print(f"[OK] Enabled single-chip (WORLD_SIZE=1) SliceBuilder init in `{os.path.relpath(fpath, tpu_exp_dir)}`")

check_code = f"""
import sys
sys.path.insert(0, {TORCHTITAN_DIR!r})
try:
    from torchtitan.experiments.tpu.components.datasets import CHAT_DATASETS
except ModuleNotFoundError:
    from torchtitan.experiments.tpu.datasets import CHAT_DATASETS
from torchtitan.experiments.tpu.transformers_modeling_backend.config_registry import hf_sft_full_max
cfg = hf_sft_full_max()
assert 'chat_json' in CHAT_DATASETS, 'Expected chat_json in CHAT_DATASETS'
model_spec = getattr(cfg, 'model_spec', None)
adapter = getattr(model_spec, 'state_dict_adapter', None)
model_cfg = getattr(model_spec, 'model', None)
print('[OK] Verified upstream `hf_sft_full_max` recipe & `chat_json` dataset loader in TorchTitan:')
print('  - state_dict_adapter : ' + str(getattr(adapter, '__name__', adapter)))
print('  - attn_mask_type     : ' + str(getattr(model_cfg, 'attn_mask_type', 'n/a')))
print('  - supported datasets : ' + str(sorted(CHAT_DATASETS.keys())))
"""
res = subprocess.run([sys.executable, "-c", check_code], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    err_tail = "\n".join((res.stderr or res.stdout).strip().splitlines()[-25:])
    raise RuntimeError(
        "TorchTitan `hf_sft_full_max` / `chat_json` check failed. Make sure Step 1 completed "
        "successfully (torch_tpu + torchtitan requirements installed).\n--- Error ---\n" + err_tail
    )

### Step 4: Run Fine-Tuning on TPU

Launches full-parameter supervised fine-tuning in `bfloat16` precision on TPU via `torchrun`.

In [ ]:
# @title { display-mode: "form" }
import glob
import os
import socket
import subprocess
import sys

TRAIN_STEPS = 150  # @param {type:"integer"}
MAX_CONTEXT_LENGTH = 2048  # @param {type:"integer"}
VAL_FREQ = 50  # @param {type:"integer"}
TRAIN_ASSETS_DIR = globals().get("TRAIN_ASSETS_DIR", "/content/functiongemma_train_assets")
DUMP_FOLDER = "/content/torchtitan_functiongemma_outputs"

TORCHTITAN_DIR = "/content/torchtitan"
if TORCHTITAN_DIR not in sys.path:
    sys.path.insert(0, TORCHTITAN_DIR)

try:
    from torchtitan.experiments.tpu.distributed import local_tpu_device
except ImportError:
    from torchtitan.experiments.tpu import local_tpu_device

tpu_chip, detected_chips = local_tpu_device.get_local_chips()
vfio_devs = [p for p in glob.glob("/dev/vfio/[0-9]*") if os.path.basename(p).isdigit()]
accel_devs = glob.glob("/dev/accel[0-9]*")
nproc_per_node = detected_chips or len(vfio_devs) or len(accel_devs) or 1
tokens_per_microbatch = 2048 if nproc_per_node == 1 else 4096

default_topo = {1: "1,1,1", 2: "1,2,1", 4: "2,2,1", 8: "2,4,1"}.get(nproc_per_node, "1,1,1")
chips_per_host_bounds = "1,1,1"

socks, sb_ports = [], []
for _ in range(nproc_per_node):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(("localhost", 0))
    sb_ports.append(s.getsockname()[1])
    socks.append(s)
for s in socks:
    s.close()

sb_addresses = ",".join(f"localhost:{p}" for p in sb_ports)
tpu_topology = os.environ.get("TORCH_TPU_TOPOLOGY") or default_topo

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc_per_node}",
    "-m",
    "torchtitan.experiments.tpu.train",
    "--module=torchtitan.experiments.tpu.transformers_modeling_backend",
    "--config=hf_sft_full_max",
    f"--dump_folder={DUMP_FOLDER}",
    f"--hf_model={TRAIN_ASSETS_DIR}",
    f"--hf_assets_path={TRAIN_ASSETS_DIR}",
    "--dataloader.dataset=chat_json",
    f"--dataloader.dataset_path={SFT_DATA_DIR}/train.jsonl",
    "--dataloader.prompt_field=question",
    "--dataloader.response_field=answer",
    "--validator.dataloader.dataset=chat_json",
    f"--validator.dataloader.dataset_path={SFT_DATA_DIR}/val.jsonl",
    "--validator.dataloader.prompt_field=question",
    "--validator.dataloader.response_field=answer",
    f"--validator.freq={VAL_FREQ}",
    "--validator.steps=5",
    "--lr_scheduler.warmup_steps=5",
    "--training.dtype=bfloat16",
    "--training.mixed_precision_param=bfloat16",
    "--training.mixed_precision_reduce=bfloat16",
    f"--training.steps={TRAIN_STEPS}",
    f"--training.max_context_length={MAX_CONTEXT_LENGTH}",
    f"--training.num_tokens_per_microbatch_per_dp_rank={tokens_per_microbatch}",
    "--checkpoint.last_save_in_hf",
    "--checkpoint.export_dtype=bfloat16",
]

print(f"Launching TorchTitan TPU fine-tuning of FunctionGemma 270M on {nproc_per_node} chip(s)...")
subprocess.run(["fuser", "-k", "/dev/vfio/0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if os.path.exists("/tmp/libtpu_lockfile"):
    os.remove("/tmp/libtpu_lockfile")

env = os.environ.copy()
env["PYTHONPATH"] = f"{TORCHTITAN_DIR}:{env.get('PYTHONPATH', '')}"
env["TORCH_TPU_SLICEBUILDER_ADDRESSES"] = sb_addresses
env["TORCH_TPU_TOPOLOGY"] = tpu_topology
env["TORCH_DYNAMO_AUTOMATIC_DYNAMIC_SHAPES"] = "0"
if nproc_per_node == 1:
    env["CLOUD_TPU_TASK_ID"] = "0"
    env["TPU_VISIBLE_CHIPS"] = "0"
    env["TPU_CHIPS_PER_HOST_BOUNDS"] = chips_per_host_bounds
    env["TPU_HOST_BOUNDS"] = tpu_topology
    env["TPU_PROCESS_ADDRESSES"] = sb_addresses
    env["TPU_PROCESS_PORT"] = str(sb_ports[0])

import collections
log_tail = collections.deque(maxlen=80)
proc = subprocess.Popen(cmd, cwd=TORCHTITAN_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    log_tail.append(line)
    if any(k in line for k in ("step:", "validate step:", "Total parameter count", "Saving the checkpoint", "Training completed")):
        print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(
        "TorchTitan TPU training exited with non-zero status. Last log lines:\n" + "".join(log_tail)
    )
print(f"\n[OK] TorchTitan TPU fine-tuning finished! Output folder: {DUMP_FOLDER}")

### Step 5: Assemble Hugging Face Checkpoint

Combines the fine-tuned language-model weights from the last training step with the tokenizer and configuration files into `/content/functiongemma_270m_tpu_finetuned_hf`. If uploading to Hugging Face Hub, set `HF_REPO_ID` and an individual `HF_SUBFOLDER` so your run does not overwrite shared files.

In [ ]:
# @title { display-mode: "form" }
import glob
import os
import shutil
import subprocess
import sys

UPLOAD_HF_CHECKPOINT_TO_HUB = False  # @param {type:"boolean"}
HF_REPO_ID = "your-hf-username/functiongemma-270m-torchtpu-litertlm"  # @param {type:"string"}
HF_SUBFOLDER = "functiongemma_270m_tpu_finetuned_hf"  # @param {type:"string"}

FINAL_HF_CHECKPOINT_DIR = "/content/functiongemma_270m_tpu_finetuned_hf"
step_ckpt_dir = os.path.join(DUMP_FOLDER, "checkpoint", f"step-{TRAIN_STEPS}")

merge_script = f"""
import glob, os, shutil
from safetensors import safe_open
from safetensors.torch import save_file

step_ckpt_dir = {step_ckpt_dir!r}
base_model_dir = {BASE_MODEL_DIR!r}
output_dir = {FINAL_HF_CHECKPOINT_DIR!r}
os.makedirs(output_dir, exist_ok=True)

shard_files = sorted(glob.glob(os.path.join(step_ckpt_dir, "model-*.safetensors")))
finetuned_st_files = shard_files or sorted(glob.glob(os.path.join(step_ckpt_dir, "*.safetensors")))
if not finetuned_st_files:
    finetuned_st_files = sorted(glob.glob(os.path.join(step_ckpt_dir, "sharded", "*.safetensors")))
if not finetuned_st_files:
    raise FileNotFoundError(f"No .safetensors files found in {{step_ckpt_dir}}.")

print(f"Loading fine-tuned weights from {{finetuned_st_files}} ...")
finetuned_tensors = {{}}
for st_path in finetuned_st_files:
    with safe_open(st_path, framework="pt", device="cpu") as f:
        for k in f.keys():
            finetuned_tensors[k] = f.get_tensor(k)
print(f"[OK] Loaded {{len(finetuned_tensors)}} fine-tuned language-model tensors.")

for fname in os.listdir(base_model_dir):
    if fname.endswith(".safetensors") or fname == "model.safetensors.index.json":
        continue
    src_path = os.path.join(base_model_dir, fname)
    if os.path.isfile(src_path):
        shutil.copy2(src_path, os.path.join(step_ckpt_dir, fname))
        shutil.copy2(src_path, os.path.join(output_dir, fname))

base_st_files = sorted(glob.glob(os.path.join(base_model_dir, "*.safetensors")))
merged_tensors = {{}}
replaced_count, frozen_copied_count = 0, 0
for st_path in base_st_files:
    with safe_open(st_path, framework="pt", device="cpu") as f:
        for k in f.keys():
            if k in finetuned_tensors:
                merged_tensors[k] = finetuned_tensors[k]
                replaced_count += 1
            else:
                merged_tensors[k] = f.get_tensor(k)
                frozen_copied_count += 1

for k, v in finetuned_tensors.items():
    if k not in merged_tensors:
        merged_tensors[k] = v

out_safetensors_path = os.path.join(output_dir, "model.safetensors")
save_file(merged_tensors, out_safetensors_path, metadata={{"format": "pt"}})
shutil.copy2(out_safetensors_path, os.path.join(step_ckpt_dir, "model.safetensors"))

size_mb = os.path.getsize(out_safetensors_path) / (1024 * 1024)
print(f"[OK] Assembled HF checkpoint in {{step_ckpt_dir}} and {{output_dir}}:")
print(f"  - Fine-tuned language-model tensors : {{replaced_count}}")
print(f"  - Frozen tower tensors copied       : {{frozen_copied_count}}")
print(f"  - Total tensors                     : {{len(merged_tensors)}} ({{size_mb:.1f}} MB)")
"""
subprocess.run([sys.executable, "-c", merge_script], check=True)

if UPLOAD_HF_CHECKPOINT_TO_HUB and HF_REPO_ID and not HF_REPO_ID.startswith("your-hf-username/"):
    from huggingface_hub import HfApi
    api = HfApi(token=globals().get("hf_token"))
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    subfolder = HF_SUBFOLDER.strip("/") or "functiongemma_270m_tpu_finetuned_hf"
    api.upload_folder(
        folder_path=FINAL_HF_CHECKPOINT_DIR,
        path_in_repo=subfolder,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"[OK] Uploaded fine-tuned HF checkpoint to https://huggingface.co/{HF_REPO_ID}/tree/main/{subfolder}")

## Part B — Convert to `.litertlm` & Verify with `litert_lm.Engine`

### Step 6: Export Base & Fine-Tuned `.litertlm` Models (`llm_model_type: function_gemma`)

Converts both the Base and Fine-Tuned Hugging Face checkpoints into `.litertlm` packages using `litert_torch.generative.export_hf` with **`litert_lm_model_type_override="function_gemma"`** so LiteRT-LM selects `FunctionGemmaDataProcessor` (`llm_model_type { function_gemma {} }`) and stops at `<start_function_response>`.

**Quantization Recipe Trade-offs (`QUANTIZATION_RECIPE`)**:
- **`dynamic_wi8_afp32` (8-bit dynamic weights, `float32` activations — Default)**: Higher precision (~285 MB `.litertlm`). Recommended for compact models like FunctionGemma 270M where strict tool-call syntax and argument accuracy are sensitive to weight quantization.
- **`dynamic_wi4b32_afp32` (4-bit blockwise weights with block size 32, `float32` activations)**: Smaller memory footprint (~170 MB `.litertlm`, ~40% smaller) and lower memory bandwidth on edge devices, trading a slight reduction in tool-call schema fidelity for smaller size.

In [ ]:
# @title { display-mode: "form" }
import importlib.metadata as md
import os
import shutil
import subprocess
import sys

QUANTIZATION_RECIPE = "dynamic_wi8_afp32"  # @param ["dynamic_wi8_afp32", "dynamic_wi4b32_afp32"]
REFERENCE_HF_REPO_ID = "litert-community/functiongemma-270m-verified-litertlm"

BASE_MODEL_DIR = globals().get("BASE_MODEL_DIR", "/content/base_functiongemma_hf")
FINAL_HF_CHECKPOINT_DIR = globals().get("FINAL_HF_CHECKPOINT_DIR", "/content/functiongemma_270m_tpu_finetuned_hf")
BASE_LITERTLM_DIR = "/content/functiongemma_270m_base_litertlm"
TUNED_LITERTLM_DIR = "/content/functiongemma_270m_tpu_finetuned_litertlm"

if not os.path.isfile(os.path.join(FINAL_HF_CHECKPOINT_DIR, "model.safetensors")):
    from huggingface_hub import snapshot_download
    os.makedirs(FINAL_HF_CHECKPOINT_DIR, exist_ok=True)
    tmp_dl = snapshot_download(
        repo_id=REFERENCE_HF_REPO_ID,
        allow_patterns=["functiongemma_270m_tpu_finetuned_hf/*"],
    )
    sub_dir = os.path.join(tmp_dl, "functiongemma_270m_tpu_finetuned_hf")
    for fname in os.listdir(sub_dir):
        shutil.copy2(os.path.join(sub_dir, fname), os.path.join(FINAL_HF_CHECKPOINT_DIR, fname))
    print(f"[OK] Restored fine-tuned HF checkpoint to {FINAL_HF_CHECKPOINT_DIR}")

# LiteRT-LM's FunctionGemmaDataProcessor passes tools to the chat template as pre-formatted
# "declaration:..." strings; let the template's format_function_declaration macro accept them.
import re
import jinja2

FC_STR_START = "{%- if tool_data is string -%}{{- tool_data -}}{%- else -%}"
FC_STR_END = "{%- endif -%}{#- fc-string-tool -#}"


def patch_fc_string_tools(model_dir: str) -> None:
    jinja_path = os.path.join(model_dir, "chat_template.jinja")
    if not os.path.isfile(jinja_path):
        return
    with open(jinja_path, "r", encoding="utf-8") as f:
        tmpl = f.read()
    hdr = "{% macro format_function_declaration(tool_data) -%}"
    hdr_idx = tmpl.find(hdr)
    if hdr_idx == -1 or FC_STR_END in tmpl:
        return
    body_start = hdr_idx + len(hdr)
    end_match = re.compile(r"\{%-?\s*endmacro").search(tmpl, body_start)
    if not end_match:
        return
    body = tmpl[body_start:end_match.start()]
    if FC_STR_START not in body:
        body = body.replace("declaration:", FC_STR_START + "declaration:", 1)
    patched = tmpl[:body_start] + body + FC_STR_END + tmpl[end_match.start():]
    jinja2.Environment().parse(patched)
    with open(jinja_path, "w", encoding="utf-8") as f:
        f.write(patched)
    print(f"[OK] Chat template in {model_dir} accepts LiteRT-LM pre-formatted tool declarations")


for d in (BASE_MODEL_DIR, FINAL_HF_CHECKPOINT_DIR):
    patch_fc_string_tools(d)

abi_pins = []
for pkg in ("numpy", "pyarrow", "pillow"):
    try:
        abi_pins.append(f"{pkg}=={md.version(pkg)}")
    except Exception:
        pass

print("Installing tf-nightly, litert-torch, ai-edge-quantizer, and litert-lm...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "--upgrade-strategy", "only-if-needed",
        "tf-nightly", "litert-torch", "ai-edge-quantizer",
        "litert-lm", "litert-lm-builder", "litert-converter", "qrcode[pil]",
        *abi_pins,
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "typing-extensions>=4.14.0"], check=True)

export_script = f"""
import logging, os, shutil
import numpy as np
from ai_edge_quantizer.algorithms.uniform_quantize import uniform_quantize_tensor as uqt
from litert_torch.generative.export_hf import export as export_hf
from litert_torch.generative.export_hf.core import litert_lm_builder as lmb_core
from litert_torch.generative.export_hf.model_ext import patches as patches_lib
import litert_torch.generative.export_hf.model_ext.gemma3.patch as _

logging.getLogger("transformers").setLevel(logging.CRITICAL)

# Newer litert-lm-builder releases removed `TfLiteModelType` (model types are now passed as strings such as
# "prefill_decode"); map the enum-style attributes litert-torch still uses onto those strings.
class _ModelTypeStr(str):
    @property
    def name(self):
        return self.upper()
    @property
    def value(self):
        return str(self)
class _TfLiteModelTypeCompat(type):
    def __getattr__(cls, attr):
        if attr.startswith("_"):
            raise AttributeError(attr)
        return _ModelTypeStr(attr.lower())
class _TfLiteModelTypeShim(metaclass=_TfLiteModelTypeCompat):
    pass
_builder_mods = [getattr(lmb_core, "litertlm_builder", None)]
try:
    import litert_lm_builder as _pip_builder
    _builder_mods.append(_pip_builder)
except ImportError:
    pass
for _bm in _builder_mods:
    if _bm is not None and not hasattr(_bm, "TfLiteModelType"):
        _bm.TfLiteModelType = _TfLiteModelTypeShim

# Register function_gemma in Gemma3 patch registries so litert_lm_model_type_override="function_gemma" applies Gemma3 RMSNorm/MLP patches
if "gemma3_text" in patches_lib._CONTEXT_REGISTRY:
    patches_lib._CONTEXT_REGISTRY["function_gemma"] = patches_lib._CONTEXT_REGISTRY["gemma3_text"]
if "gemma3_text" in patches_lib._MODEL_PATCH_REGISTRY:
    patches_lib._MODEL_PATCH_REGISTRY["function_gemma"] = patches_lib._MODEL_PATCH_REGISTRY["gemma3_text"]

# Ensure LlmMetadata includes <start_function_response> stop token and llm_model_type: function_gemma
orig_build_llm_metadata = lmb_core.build_llm_metadata
def _fg_build_llm_metadata(*args, **kwargs):
    llm_metadata = orig_build_llm_metadata(*args, **kwargs)
    llm_metadata.llm_model_type.CopyFrom(
        lmb_core.llm_model_type_pb2.LlmModelType(
            function_gemma=lmb_core.llm_model_type_pb2.FunctionGemma()
        )
    )
    existing_stops = {{st.token_str for st in llm_metadata.stop_tokens if st.HasField("token_str")}}
    for req_stop in ("<end_of_turn>", "<start_function_response>"):
        if req_stop not in existing_stops:
            tu = llm_metadata.stop_tokens.add()
            tu.token_str = req_stop
    return llm_metadata
lmb_core.build_llm_metadata = _fg_build_llm_metadata

orig_zp_scale_fn = uqt.tensor_zp_scale_from_min_max
def _safe_zp_scale_from_min_max(*args, **kwargs):
    zp, scale = orig_zp_scale_fn(*args, **kwargs)
    granularity = kwargs.get("granularity", args[4] if len(args) > 4 else None)
    if granularity is not None and uqt.is_blockwise(granularity):
        scale = np.maximum(scale, np.float32(6.58e-5))
    return zp, scale
uqt.tensor_zp_scale_from_min_max = _safe_zp_scale_from_min_max

for d in ({BASE_LITERTLM_DIR!r}, {TUNED_LITERTLM_DIR!r}):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

print("Exporting Base FunctionGemma 270M -> {BASE_LITERTLM_DIR}/model.litertlm ...")
export_hf.export(
    model={BASE_MODEL_DIR!r},
    output_dir={BASE_LITERTLM_DIR!r},
    quantization_recipe={QUANTIZATION_RECIPE!r},
    litert_lm_model_type_override="function_gemma",
    moe_exports_implementation="eager",
)

print("Exporting Fine-Tuned FunctionGemma 270M -> {TUNED_LITERTLM_DIR}/model.litertlm ...")
export_hf.export(
    model={FINAL_HF_CHECKPOINT_DIR!r},
    output_dir={TUNED_LITERTLM_DIR!r},
    quantization_recipe={QUANTIZATION_RECIPE!r},
    litert_lm_model_type_override="function_gemma",
    moe_exports_implementation="eager",
)
print("[OK] Exported .litertlm packages ({QUANTIZATION_RECIPE}, llm_model_type=function_gemma)!")
"""
proc = subprocess.Popen([sys.executable, "-c", export_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError("LiteRT-LM export failed.")

BASE_LITERTLM_PATH = f"{BASE_LITERTLM_DIR}/model.litertlm"
TUNED_LITERTLM_PATH = f"{TUNED_LITERTLM_DIR}/model.litertlm"

### Step 7: Evaluate Base vs. Fine-Tuned on Held-Out Requests

Passes the three smart-home tools to `engine.create_conversation(tools=...)` and compares both `.litertlm` models on held-out requests that never appear in training. It reports how often each model picks the right function and how often it produces the exact call (function **and** arguments, e.g. `set_device_state(device_id="light.living_room", state="on")`).

In [ ]:
import json
import litert_lm

EVAL_N = 100  # @param {type:"integer"}
SFT_DATA_DIR = "/content/sft_data"
with open(f"{SFT_DATA_DIR}/tools.json", encoding="utf-8") as f:
    TOOLS = json.load(f)
with open(f"{SFT_DATA_DIR}/eval_toolcalls.jsonl", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f][:EVAL_N]


class SmartHomeTool(litert_lm.Tool):
    """Passes the exact OpenAPI schema used for training to LiteRT-LM."""

    def __init__(self, definition):
        self._definition = definition

    def get_tool_description(self):
        return {"type": "function", "function": self._definition}

    def execute(self, param):
        return "ok"


def first_tool_call(reply):
    calls = reply.get("tool_calls") if isinstance(reply, dict) else getattr(reply, "tool_calls", None)
    if not calls:
        return None, None
    call = calls[0]
    if not isinstance(call, dict):  # litert_lm.ToolCall-style object
        return getattr(call, "name", None), dict(getattr(call, "arguments", {}) or {})
    fn = call.get("function", call)
    args = fn.get("arguments", {})
    if isinstance(args, str):
        args = json.loads(args or "{}")
    return fn.get("name"), dict(args)


def norm_args(args):
    return {k: str(v).strip().lower() for k, v in (args or {}).items()}


cpu_backend = litert_lm.Backend.CPU() if callable(getattr(litert_lm.Backend, "CPU", None)) else litert_lm.Backend.CPU
tools = [SmartHomeTool(t) for t in TOOLS]
DEMO_EXAMPLES = eval_rows[:2]  # "Turn on the living room lights", "I want the office at 22 degrees"
results, demo_calls = {}, {}

for label, model_path in [("Base", BASE_LITERTLM_PATH), ("Fine-tuned", TUNED_LITERTLM_PATH)]:
    name_ok = exact_ok = 0
    demo_calls[label] = []
    with litert_lm.Engine(model_path=model_path, backend=cpu_backend) as engine:
        for i, row in enumerate(eval_rows):
            with engine.create_conversation(tools=tools, automatic_tool_calling=False) as conv:
                reply = conv.send_message(row["query"], max_output_tokens=64)
            name, args = first_tool_call(reply)
            correct = name == row["name"] and norm_args(args) == norm_args(row["arguments"])
            if i < len(DEMO_EXAMPLES):
                demo_calls[label].append((name, args, correct))
            name_ok += name == row["name"]
            exact_ok += correct
    results[label] = (name_ok / len(eval_rows), exact_ok / len(eval_rows))

for i, row in enumerate(DEMO_EXAMPLES):
    print("=" * 80 + f"\nPROMPT:   {row['query']}\nEXPECTED: {row['name']}({json.dumps(row['arguments'])})")
    for label in ("Base", "Fine-tuned"):
        name, args, correct = demo_calls[label][i]
        shown = f"{name}({json.dumps(args)})" if name else "(no tool call)"
        print(f"  {'✅' if correct else '❌'} {label:<11} {shown}")

print(f"\nHeld-out smart-home requests (n={len(eval_rows)}, unseen phrasings/devices):")
print(f"{'Model':<12}{'Correct function':>18}{'Exact call':>14}")
for label, (fn_acc, exact_acc) in results.items():
    print(f"{label:<12}{fn_acc:>18.1%}{exact_acc:>14.1%}")

### Step 8: Share `.litertlm` via Hugging Face Hub

Uploads the Base and Fine-Tuned `.litertlm` packages plus `tools.json` to an **individual directory** (`HF_EXPORT_SUBFOLDER`) inside your Hugging Face repository (`HF_EXPORT_REPO_ID`) so multiple runs or users never overwrite shared files. The companion CLI & Web Colab downloads them from there.

In [ ]:
# @title { display-mode: "form" }
import io
import os
import qrcode
from IPython.display import Image as IPyImage, display

UPLOAD_TO_HF_HUB = False  # @param {type:"boolean"}
HF_EXPORT_REPO_ID = "your-hf-username/functiongemma-270m-torchtpu-litertlm"  # @param {type:"string"}
HF_EXPORT_SUBFOLDER = "functiongemma_270m_torchtpu"  # @param {type:"string"}

quant_recipe = globals().get("QUANTIZATION_RECIPE", "dynamic_wi8_afp32")
subfolder = HF_EXPORT_SUBFOLDER.strip("/") or "functiongemma_270m_torchtpu"
# Base + fine-tuned models and the tool schema, so the companion CLI & Web Colab can reuse them.
uploads = {
    f"{subfolder}/functiongemma_270m_torchtpu_{quant_recipe}.litertlm": globals().get(
        "TUNED_LITERTLM_PATH", "/content/functiongemma_270m_tpu_finetuned_litertlm/model.litertlm"),
    f"{subfolder}/functiongemma_270m_base_{quant_recipe}.litertlm": globals().get(
        "BASE_LITERTLM_PATH", "/content/functiongemma_270m_base_litertlm/model.litertlm"),
    f"{subfolder}/tools.json": "/content/sft_data/tools.json",
}
tuned_path_in_repo = next(iter(uploads))

if UPLOAD_TO_HF_HUB:
    if HF_EXPORT_REPO_ID.startswith("your-hf-username/"):
        raise ValueError("Please set HF_EXPORT_REPO_ID to your own Hugging Face repository (e.g. 'username/functiongemma-270m-torchtpu-litertlm').")
    missing = [p for p in uploads.values() if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(f"Cannot upload, files not found: {missing}. Run Steps 2 and 6 first.")
    from huggingface_hub import HfApi
    api = HfApi(token=globals().get("hf_token"))
    api.create_repo(repo_id=HF_EXPORT_REPO_ID, repo_type="model", exist_ok=True)
    for path_in_repo, local_path in uploads.items():
        api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo, repo_id=HF_EXPORT_REPO_ID, repo_type="model")
        print(f"[OK] Uploaded {path_in_repo} to https://huggingface.co/{HF_EXPORT_REPO_ID}")

    download_url = f"https://huggingface.co/{HF_EXPORT_REPO_ID}/resolve/main/{tuned_path_in_repo}"
    print(f"Direct fine-tuned .litertlm URL: {download_url}")

    qr_img = qrcode.make(download_url)
    buf = io.BytesIO()
    qr_img.save(buf, format="PNG")
    display(IPyImage(data=buf.getvalue(), width=220))
else:
    print("Skipped Hugging Face upload. Check UPLOAD_TO_HF_HUB and set HF_EXPORT_REPO_ID to upload and get a download URL + QR code.")

## Next Steps: Run Your `.litertlm` Model Everywhere

Once you have exported your `.litertlm` model, you can deploy and test it across mobile, web, and desktop:

- **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**: Run, benchmark, and serve `.litertlm` models from your terminal (`pip install litert-lm && litert-lm run model.litertlm`), or open the companion **[FunctionGemma CLI & Web Playground Colab](https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_TorchTPU_CLI_and_Web.ipynb)**.
- **[LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)**: Upload your `.litertlm` package in Chrome or Edge to run 100% client-side in the browser with WebGPU acceleration.
- **[Google AI Edge Gallery App](https://developers.google.com/edge/gallery)**: Import your `.litertlm` model on Android or iOS (via your Hugging Face download URL or QR code above) for interactive on-device inference.